In [ ]:
from sentence_transformers import SentenceTransformer, util

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
sen1 = "He is reading a book in the library"
sen2 = "He is at the library reading a book"

In [ ]:
emb1 = model.encode(sen1, convert_to_tensor=True)  # 결과를 Tensor(파이토치 숫자 상자)로 받기
emb2 = model.encode(sen2, convert_to_tensor=True)

cos_sim = util.pytorch_cos_sim(emb1, emb2)

In [ ]:
print(f"두 문장의 유사도 : {cos_sim.item():.4f}") 

---

# 감정분석

---


In [ ]:
from transformers import pipeline  # 핵심 진입점 → 태스크명만으로 모델·토크나이저 자동 결합

In [ ]:
classifier = pipeline("sentiment-analysis")  # 최초 1회 ~/.cache 저장 → 이후 즉시 로드

In [ ]:
text = "I'm feeling really great today"
results = classifier(text)  # 반환 형태: [{'label': str, 'score': float}] 리스트

In [ ]:
print(f"감정 분석 결과 : {results[0]['label']}")    # 'label' 키 → 'POSITIVE' 또는 'NEGATIVE' 두 값
# → POSITIVE
print(f"감정 분석 점수 : {results[0]['score']:.4f}")  # 'score' 키 → 0~1 확률, :.4f 로 소수 4자리 포맷
# → 0.9998  (1에 가까울수록 모델 확신도 높음)

---

# 택스트 생성

---

In [ ]:
generator = pipeline("text-generation", model="gpt2")

In [ ]:
answer = input("생성 문장을 입력해주세요 : ")

In [ ]:
result = generator(
    answer,
    max_new_tokens=50,       # 추가 생성할 토큰 수 → 길수록 추론 시간 ↑
    num_return_sequences=1,  # 반환 문장 개수 → 2 이상이면 후보 비교 가능
    truncation=True          # 입력이 모델 최대 길이 초과 시 자르기
)

In [ ]:
# 4. 결과 출력 (반환은 [{'generated_text': str}] 리스트)
print(result[0]["generated_text"])

---

# 영문 요약

---

In [ ]:
summarizer = pipeline("summarization", model="t5-small")  # 약 240MB 경량 버전

In [ ]:
# 2. 요약할 원문 (영어 단락)
text = """A special 25th anniversary edition of the extraordinary
international bestseller... Santiago's journey teaches us about
the essential wisdom of listening to our hearts..."""

In [ ]:
# 3. 요약 실행 (길이 옵션 지정)
# (왜) 기본값은 모델마다 다름 → 명시 지정해야 결과 길이 예측 가능
summary = summarizer(
    text,
    min_length=20,   # 최소 토큰 수 → 너무 짧은 요약 방지
    max_length=60,   # 최대 토큰 수 → 길이 폭주 방지
    do_sample=False  # 결정적(greedy) 생성 → 매번 동일 결과
)  # 반환: [{'summary_text': str}] 리스트

In [ ]:
# 4. 결과 텍스트 추출 → 출력
sum_text = summary[0]['summary_text']  # 첫 결과의 'summary_text' 키 → 요약 문자열
print(f"요약된 문장 : {sum_text}")

---

# 한국어 번역

---

In [ ]:
from deep_translator import GoogleTranslator

In [ ]:
def trans_en_to_ko(sentence):
    """
    주어진 영어 문장을 한국어로 번역하는 함수
    """
    translated_sen = GoogleTranslator(source='en', target='ko').translate(sentence)  # source 출발 / target 도착 언어 → 구글 서버 요청 → 한국어 문자열 반환
    return translated_sen

# 3. 요약 파이프라인 생성
# (왜) 토크나이저·모델·후처리 직접 짜기 번거로움 → "summarization" 한 줄로 결합
summarizer = pipeline(
    "summarization",
    model="t5-small"   # 약 240MB 경량 영어 요약 모델
)

In [ ]:
# 4. 요약할 영어 원문 (긴 단락)
text = """
A special 25th anniversary edition of the extraordinary international bestseller, including a new Foreword by Paulo Coelho.
Combining magic, mysticism, wisdom and wonder into an inspiring tale of self-discovery, The Alchemist has become a modern classic, selling millions of copies around the world and transforming the lives of countless readers across generations.
Paulo Coelho's masterpiece tells the mystical story of Santiago, an Andalusian shepherd boy who yearns to travel in search of a worldly treasure. His quest will lead him to riches far different-and far more satisfying-than he ever imagined. Santiago's journey teaches us about the essential wisdom of listening to our hearts, of recognizing opportunity and learning to read the omens strewn along life's path, and, most importantly, to follow our dreams.
"""

In [ ]:
# 5. 요약문 생성 (T5 인코더가 문맥 압축 → 디코더가 짧은 문장 생성)
summary = summarizer(text)  # 반환: [{'summary_text': str}] 리스트

In [ ]:
# 6. 요약문 텍스트 추출
sum_text = summary[0]["summary_text"]  # 첫 결과의 'summary_text' 키 → 요약 문자열

In [ ]:
# 7. 영어 요약문 출력
print(f"### 요약된 영어 문장 : {sum_text} ###")

In [ ]:
# 8. 요약문 → 한국어 번역
# (왜) T5는 영어만 잘함 → 결과를 한국어로 옮기려면 외부 번역기 결합 필수
# kr_sum_text = GoogleTranslator(source='en', target='ko').translate(sum_text)   # ← 함수 안 쓰는 1줄 버전 (동일 동작)
kr_sum_text = trans_en_to_ko(sum_text)  # 함수 호출 → 가독성 ↑, 재사용 ↑

In [ ]:
# 9. 한국어 번역문 출력
print(f"### 번역된 한국어 문장 : {kr_sum_text} ###")